### Bonds Arbitrage

The main idea is to find mispricing, this allows (given some assumptions such as ability to short sell and no transaction fees) to exploit the mispricing and gain a risk-free profit 

By observing several bond prices and their future cash flows, we can build a system of linear equations. Solving this system gives the discount factors for different maturities. From these discount factors, we can then derive the corresponding spot yields

When no solution is possible, if the observed bond prices are inconsistent with any common set of discount factors, then at least one bond is mispriced relative to the others. In that case, an arbitrage portfolio may exist

$$
\begin{align}
P_1 &= C_{1,1}d_1 + C_{1,2}d_2 + C_{1,T}d_T \\
P_2 &= C_{2,1}d_1 + C_{2,2}d_2 + C_{2,T}d_T \\
\vdots \\
P_n &= C_{n,1}d_1 + C_{n,2}d_2 + C_{n,T}d_T \\
\end{align}
$$

$$
\begin{bmatrix}
C_{1,1} & C_{1,2} & \cdots & C_{1,T} \\
C_{2,1} & C_{2,2} & \cdots & C_{2,T} \\
\vdots & \vdots & \ddots & \vdots \\
C_{n,1} & C_{n,2} & \cdots & C_{n,T}
\end{bmatrix}
\begin{bmatrix}
d_1 \\
d_2 \\
\vdots \\
d_T
\end{bmatrix}
=
\begin{bmatrix}
P_1 \\
P_2 \\
\vdots \\
P_n
\end{bmatrix}
$$

In [1]:
import numpy as np


def analyze_bonds(cash_flows, prices, maturities):
    C = np.array(cash_flows, dtype=float)
    P = np.array(prices, dtype=float)
    maturities = np.array(maturities, dtype=float)

    print("Cash-flow matrix C:")
    print(C)

    print("\nPrice vector P:")
    print(P)

    print("\nLinear system:")
    print("C @ d = P")

    discount_factors, residuals, rank, singular_values = np.linalg.lstsq(C, P, rcond=None)

    fitted_prices = C @ discount_factors
    pricing_errors = P - fitted_prices

    print("\nEstimated discount factors:")
    for i, d in enumerate(discount_factors, start=1):
        print(f"d_{i} = {d:.6f}")

    print("\nFitted prices:")
    print(fitted_prices)

    print("\nPricing errors:")
    print(pricing_errors)

    exact_solution = np.allclose(fitted_prices, P)

    if exact_solution:
        print("\nThe system is consistent")
        print("No arbitrage detected from this set of prices")
    else:
        print("\nThe system is inconsistent")
        print("The prices imply a relative mispricing")

    yields = discount_factors ** (-1 / maturities) - 1

    print("\nSpot yields:")
    for i, y in enumerate(yields, start=1):
        print(f"y_{i} = {100 * y:.4f}%")

    return discount_factors, yields, fitted_prices, pricing_errors

#### Consider No-Arbitrage Example

| Bond | $C_1$ | $C_2$ | $C_3$ | $C_4$ | $C_5$ |  Price |
| ---: | ----: | ----: | ----: | ----: | ----: | -----: |
|    1 |   100 |     0 |     0 |     0 |     0 |  99.00 |
|    2 |     0 |   100 |     0 |     0 |     0 |  97.50 |
|    3 |     0 |     0 |   100 |     0 |     0 |  95.50 |
|    4 |     0 |     0 |     0 |   100 |     0 |  93.00 |
|    5 |     0 |     0 |     0 |     0 |   100 |  90.00 |
|    6 |     4 |   104 |     0 |     0 |     0 | 105.36 |
|    7 |     5 |     5 |   105 |     0 |     0 | 110.10 |
|    8 |     3 |     3 |     3 |   103 |     0 | 104.55 |
|    9 |     6 |     6 |     6 |     6 |   106 | 118.50 |
|   10 |     2 |     2 |     2 |     2 |   102 |  99.50 |


In [2]:
# No Arbitrage example
cash_flows_no_arbitrage = [
    [100, 0],
    [0, 100],
    [5, 105],
]

prices_no_arbitrage = [
    95,
    90,
    99.25,
]

maturities = [
    1,
    2,
]

analyze_bonds(
    cash_flows=cash_flows_no_arbitrage,
    prices=prices_no_arbitrage,
    maturities=maturities,
)

Cash-flow matrix C:
[[100.   0.]
 [  0. 100.]
 [  5. 105.]]

Price vector P:
[95.   90.   99.25]

Linear system:
C @ d = P

Estimated discount factors:
d_1 = 0.950000
d_2 = 0.900000

Fitted prices:
[95.   90.   99.25]

Pricing errors:
[-1.42108547e-14 -2.84217094e-14 -4.26325641e-14]

The system is consistent
No arbitrage detected from this set of prices

Spot yields:
y_1 = 5.2632%
y_2 = 5.4093%


(array([0.95, 0.9 ]),
 array([0.05263158, 0.05409255]),
 array([95.  , 90.  , 99.25]),
 array([-1.42108547e-14, -2.84217094e-14, -4.26325641e-14]))

In [ ]:
# Arbitrage example
cash_flows_arbitrage = [
    [100, 0],
    [0, 100],
    [5, 105],
]

prices_arbitrage = [
    95,
    90,
    97,
]

analyze_bonds(
    cash_flows=cash_flows_arbitrage,
    prices=prices_arbitrage,
    maturities=maturities,
)

In [ ]:
# Here Bond C is too cheap.
# Its fair price implied by Bond A and Bond B is:

d1 = 95 / 100
d2 = 90 / 100

fair_price_C = 5 * d1 + 105 * d2
market_price_C = 97

mispricing = fair_price_C - market_price_C

print(f"Fair price of Bond C: {fair_price_C:.2f}")
print(f"Market price of Bond C: {market_price_C:.2f}")
print(f"Mispricing: {mispricing:.2f}")